In [1]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:120b-cloud")
print("Model is ready")

Model is ready


In [4]:
# Example - Prompt without output parser
from langchain_core.prompts import PromptTemplate

user_prompt_template = PromptTemplate(
    template="Give me list of only names of 5 important cities in India"
)

user_prompt = user_prompt_template.format()

response = model.invoke(user_prompt)
print(type(response))
print(response)

<class 'langchain_core.messages.ai.AIMessage'>
content='Mumbai  \nDelhi  \nBengaluru  \nChennai  \nKolkata' additional_kwargs={} response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-09-09T03:30:51.493470875Z', 'done': True, 'done_reason': 'stop', 'total_duration': 365034122, 'load_duration': None, 'prompt_eval_count': 80, 'prompt_eval_duration': None, 'eval_count': 79, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'} id='lc_run--01a08437-e4c7-73c1-84d7-842c4ad9397f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 80, 'output_tokens': 79, 'total_tokens': 159}


In [5]:
from langchain_core.output_parsers import StrOutputParser
user_prompt_template=PromptTemplate(template="Write a kannada peom on Karnataka Kaveri")

parser=StrOutputParser()
chain =user_prompt_template|model|parser
response= chain.invoke({})
print(response)

ಕರವಳ ತೀರದಲ್ಲಿ ಕಾವೇರಿ ನಡಿವೆಯು  
ಹಳ್ಳಿ ಹೃದಯದಲ್ಲಿ ಹೊಳೆಯುವ ಹೊಂಗೆ ಮಂಜು.  

ಕೆಂಪು ರಂಗಿನ ನದೀ ಜಲದ ದಾಂಪತ್ಯ,  
ಮರುಭೂಮಿಯ ಕಣಿವೆಗಳಲ್ಲಿ ಹಸಿರು ಹೊತ್ತು.  

ಬಂಗಾರದ ಹಕ್ಕಿ ಕಿರಣ – ತೋಟದ ಹಿನ್ನೀರು,  
ಮರಳಿ ಮೇದಾನದಲ್ಲಿ ಬಾಪರಿನ ಮೃದುಮನೆ.  

ಮಣ್ಣಿನ ಸ್ನೇಹದಲ್ಲಿ ನೂರಾರು ಕವಿತೆ,  
ಬರಿದ ತಂಗುಗಳೆಲ್ಲ ಕಾವೇರಿ ಹಾಡು.  

ಮಲೆಗಳ ಎತ್ತರದಲ್ಲಿ ಹೊಳೆಯುವ ಕಂಚು,  
ಕೇರಳ‑ಕರ್ನಾಟಕ ಭೂಮಿಯ ಏಕಮಾತು.  

ಹಾರಿದೊಳಗು ನದೀಕಂದ, ಕನಸಿನ ರೇಖೆಗೆ,  
ಕಾವೇರಿ ನೀರಿನ ಚಾವಣಿ – ಪ್ರಾಣದ ಪ್ರೀತಿ.  

*ಇದು ಕರ್ನಾಟಕದ ಹೃದಯ ನಾಡು,  
ಅದರ ಹರಿವುದಲ್ಲಿ ನೂರಾರು ಕನಸುಗಳ ಸರಳ ಮರುಕ.*


In [6]:
from langchain_core.output_parsers import ListOutputParser
from langchain_core.prompts import PromptTemplate

class CommaSeparatedCities(ListOutputParser):
    def parse(self, text: str) -> list[str]:
        return [item.strip() for item in text.split(",")]

prompt = PromptTemplate(
    template="Give me comma separated list of only names in English language of top 5 populous cities in the world",
    input_variables=[]
)

list_parser = CommaSeparatedCities()

chain = prompt | model | list_parser

response = chain.invoke({})
print(response)

['Tokyo', 'Delhi', 'Shanghai', 'São\u202fPaulo', 'Mexico City']


In [7]:
# Example - json output parser with PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

prompt = PromptTemplate(
    template="Extract the person's details from the text below.\n"
    "Text:\n"
    "Tom having age 25 years works with Zensar Technologies at Pune location"
)
# Output Schema
class Person(BaseModel):
    name: str = Field("Peron's name"),
    age: int = Field("Person's age"),
    company: str = ("Company name"),
    location: str = Field("Office Location")

json_parser = JsonOutputParser(pydantic_object=Person)

chain = prompt | model | json_parser
response = chain.invoke({})
print(response)


{'name': 'Tom', 'age': 25, 'employer': 'Zensar Technologies', 'location': 'Pune'}


In [10]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel ,Field
from langchain_core.prompts import ChatPromptTemplate
# Output Schema
class Person(BaseModel):
    name: str = Field("Peron's name"),
    age: int = Field("Person's age"),
    company: str = ("Company name"),
    location: str = Field("Office Location")

json_parser = JsonOutputParser(pydantic_object=Person)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful AI assistant"),
    ("user", "#Format: {format_instructions}\n\nQuestion: {question}")
]).partial(format_instructions=json_parser.get_format_instructions())


chain = prompt | model | json_parser
response = chain.invoke({"question": "sagar m having age 25 years works with Zensar Technologies at Pune location"})
print(response)

c:\Users\kakno\AppData\Local\Programs\Python\Python314\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=False, default="Peron's name"),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Users\kakno\AppData\Local\Programs\Python\Python314\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=False, default="Person's age"),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


{'name': 'sagar m', 'age': 25, 'company': 'Zensar Technologies', 'location': 'Pune'}


In [15]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
#Output Schema

prompt = PromptTemplate(
    template="Extract the person's details from the text below.\n"
    "Text:\n"
    "Tom having age 25 years works with Zensar Technologies at Pune location"
)
#Output Schema
class Person(BaseModel):
     name: str = Field("Peron's name"),
     age: int = Field("Person's age"),
     company: str = ("Company name"),
     location: str = Field("Office Location")
output_parser=PydanticOutputParser(pydantic_object=Person)
chain = prompt | model | output_parser
response = chain.invoke({})
print(response)

name='Tom' age=25 company='Zensar Technologies' location='Pune'


In [17]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel ,Field
from langchain_core.prompts import ChatPromptTemplate
# Output Schema
class Person(BaseModel):
    name: str = Field("Peron's name"),
    age: int = Field("Person's age"),
    company: str = ("Company name"),
    location: str = Field("Office Location")

pydantic_outputParser= PydanticOutputParser(pydantic_object=Person)
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful AI assistant"),
    ("user", "#Format: {format_instructions}\n\nQuestion: {question}")
]).partial(format_instructions=pydantic_outputParser.get_format_instructions())


chain = prompt | model | pydantic_outputParser
response = chain.invoke({"question": "sagar m having age 25 years works with Zensar Technologies at Pune location"})
print(response)

c:\Users\kakno\AppData\Local\Programs\Python\Python314\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=False, default="Peron's name"),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\Users\kakno\AppData\Local\Programs\Python\Python314\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=False, default="Person's age"),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


name='sagar m' age=25 company='Zensar Technologies' location='Pune'
